In [1]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset


# ============================================================
# CONFIG
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

EMBEDDING_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "embeddings"
    / "clip"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "fusion_experiments"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "fusion_experiments"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 8
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ============================================================
# DEVICE
# ============================================================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)


# ============================================================
# LOAD DATA
# ============================================================

def load_split(split_name):
    image = np.load(
        EMBEDDING_ROOT / f"{split_name}_image_embeddings.npy"
    ).astype(np.float32)

    text = np.load(
        EMBEDDING_ROOT / f"{split_name}_text_embeddings.npy"
    ).astype(np.float32)

    df = pd.read_parquet(
        MODEL_INPUT_ROOT / f"{split_name}.parquet"
    ).reset_index(drop=True)

    return image, text, df


X_train_image, X_train_text, train_df = load_split("train")
X_val_image, X_val_text, validation_df = load_split("validation")
X_test_image, X_test_text, test_df = load_split("test")


# ============================================================
# STRUCTURED FEATURES
# ============================================================

for df in [train_df, validation_df, test_df]:
    df["reviews_log1p"] = np.log1p(
        pd.to_numeric(df["reviews"], errors="coerce").fillna(0)
    )

    df["bought_log1p"] = np.log1p(
        pd.to_numeric(
            df["boughtInLastMonth"],
            errors="coerce",
        ).fillna(0)
    )

numeric_columns = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",
]

categorical_columns = [
    "category_name",
    "price_band",
]

preprocessor = ColumnTransformer(
    [
        (
            "num",
            StandardScaler(),
            numeric_columns,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_columns,
        ),
    ]
)

X_train_structured = preprocessor.fit_transform(
    train_df
).astype(np.float32)

X_val_structured = preprocessor.transform(
    validation_df
).astype(np.float32)

X_test_structured = preprocessor.transform(
    test_df
).astype(np.float32)

joblib.dump(
    preprocessor,
    OUTPUT_ROOT / "structured_preprocessor.joblib",
)

STRUCTURED_DIM = X_train_structured.shape[1]

print("Structured dim:", STRUCTURED_DIM)


# ============================================================
# TARGETS
# ============================================================

y_train = train_df["price"].astype(np.float32).to_numpy()
y_val = validation_df["price"].astype(np.float32).to_numpy()
y_test = test_df["price"].astype(np.float32).to_numpy()

y_train_log = np.log1p(y_train).astype(np.float32)
y_val_log = np.log1p(y_val).astype(np.float32)
y_test_log = np.log1p(y_test).astype(np.float32)


# ============================================================
# DATASET
# ============================================================

class PriceDataset(Dataset):
    def __init__(
        self,
        image,
        text,
        structured,
        target,
    ):
        self.image = torch.from_numpy(image)
        self.text = torch.from_numpy(text)
        self.structured = torch.from_numpy(structured)
        self.target = torch.from_numpy(target)

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):
        return (
            self.image[idx],
            self.text[idx],
            self.structured[idx],
            self.target[idx],
        )


train_dataset = PriceDataset(
    X_train_image,
    X_train_text,
    X_train_structured,
    y_train_log,
)

val_dataset = PriceDataset(
    X_val_image,
    X_val_text,
    X_val_structured,
    y_val_log,
)

test_dataset = PriceDataset(
    X_test_image,
    X_test_text,
    X_test_structured,
    y_test_log,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


# ============================================================
# MODEL 1: STRUCTURED-ONLY MLP
# ============================================================

class StructuredOnlyMLP(nn.Module):
    def __init__(self, structured_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(structured_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1),
        )

    def forward(
        self,
        image,
        text,
        structured,
    ):
        return self.network(
            structured
        ).squeeze(1)


# ============================================================
# MODEL 2: GATED MULTIMODAL FUSION
# ============================================================

class GatedMultimodalPriceModel(nn.Module):
    def __init__(
        self,
        image_dim,
        text_dim,
        structured_dim,
    ):
        super().__init__()

        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.structured_encoder = nn.Sequential(
            nn.Linear(structured_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.gate = nn.Sequential(
            nn.Linear(
                128 * 3,
                128,
            ),
            nn.ReLU(),
            nn.Linear(
                128,
                3,
            ),
            nn.Softmax(dim=1),
        )

        self.regressor = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1),
        )

    def forward(
        self,
        image,
        text,
        structured,
    ):
        image_h = self.image_encoder(
            image
        )

        text_h = self.text_encoder(
            text
        )

        structured_h = (
            self.structured_encoder(
                structured
            )
        )

        combined = torch.cat(
            [
                image_h,
                text_h,
                structured_h,
            ],
            dim=1,
        )

        gates = self.gate(
            combined
        )

        fused = (
            gates[:, 0:1] * image_h
            + gates[:, 1:2] * text_h
            + gates[:, 2:3] * structured_h
        )

        prediction = self.regressor(
            fused
        ).squeeze(1)

        return prediction, gates


# ============================================================
# METRICS
# ============================================================

def price_metrics(
    actual_log,
    predicted_log,
):
    actual = np.expm1(
        actual_log
    )

    predicted = np.expm1(
        predicted_log
    )

    predicted = np.clip(
        predicted,
        0,
        None,
    )

    return {
        "mae": mean_absolute_error(
            actual,
            predicted,
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                actual,
                predicted,
            )
        ),
        "median_ae": median_absolute_error(
            actual,
            predicted,
        ),
        "r2": r2_score(
            actual,
            predicted,
        ),
    }


# ============================================================
# GENERIC TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    model_name,
    gated=False,
):
    model = model.to(
        DEVICE
    )

    criterion = nn.HuberLoss(
        delta=1.0
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
        )
    )

    best_mae = float("inf")
    no_improvement = 0

    history = []

    best_model_path = (
        OUTPUT_ROOT
        / f"{model_name}_best.pth"
    )

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        # =========================
        # TRAIN
        # =========================
        model.train()

        train_loss = 0.0

        for (
            image,
            text,
            structured,
            target,
        ) in train_loader:

            image = image.to(DEVICE)
            text = text.to(DEVICE)
            structured = structured.to(DEVICE)
            target = target.to(DEVICE)

            optimizer.zero_grad()

            if gated:
                prediction, _ = model(
                    image,
                    text,
                    structured,
                )
            else:
                prediction = model(
                    image,
                    text,
                    structured,
                )

            loss = criterion(
                prediction,
                target,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0,
            )

            optimizer.step()

            train_loss += (
                loss.item()
                * len(target)
            )

        train_loss /= len(
            train_dataset
        )

        # =========================
        # VALIDATION
        # =========================
        model.eval()

        val_predictions = []
        val_targets = []
        val_loss = 0.0

        gate_values = []

        with torch.inference_mode():

            for (
                image,
                text,
                structured,
                target,
            ) in val_loader:

                image = image.to(DEVICE)
                text = text.to(DEVICE)
                structured = structured.to(DEVICE)
                target = target.to(DEVICE)

                if gated:
                    prediction, gates = model(
                        image,
                        text,
                        structured,
                    )

                    gate_values.append(
                        gates.cpu().numpy()
                    )

                else:
                    prediction = model(
                        image,
                        text,
                        structured,
                    )

                loss = criterion(
                    prediction,
                    target,
                )

                val_loss += (
                    loss.item()
                    * len(target)
                )

                val_predictions.append(
                    prediction
                    .cpu()
                    .numpy()
                )

                val_targets.append(
                    target
                    .cpu()
                    .numpy()
                )

        val_loss /= len(
            val_dataset
        )

        val_predictions = np.concatenate(
            val_predictions
        )

        val_targets = np.concatenate(
            val_targets
        )

        metrics = price_metrics(
            val_targets,
            val_predictions,
        )

        scheduler.step(
            val_loss
        )

        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_mae": metrics["mae"],
                "val_rmse": metrics["rmse"],
                "val_r2": metrics["r2"],
                "learning_rate": current_lr,
            }
        )

        print(
            f"{model_name} | "
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val MAE {metrics['mae']:.4f} | "
            f"Val RMSE {metrics['rmse']:.4f} | "
            f"Val R² {metrics['r2']:.4f} | "
            f"LR {current_lr:.6f}"
        )

        if metrics["mae"] < best_mae:

            best_mae = metrics["mae"]

            no_improvement = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict":
                        model.state_dict(),
                    "best_val_mae":
                        best_mae,
                },
                best_model_path,
            )

            print(
                f"  ✅ Best {model_name} "
                f"saved: {best_mae:.4f}"
            )

        else:
            no_improvement += 1

        if (
            no_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Early stopping "
                f"{model_name}"
            )
            break

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        REPORT_ROOT
        / f"{model_name}_history.csv",
        index=False,
    )

    return (
        best_model_path,
        history_df,
    )


# ============================================================
# TRAIN STRUCTURED-ONLY MLP
# ============================================================

structured_model = StructuredOnlyMLP(
    structured_dim=STRUCTURED_DIM
)

structured_model_path, structured_history = (
    train_model(
        structured_model,
        "structured_mlp",
        gated=False,
    )
)


# ============================================================
# TRAIN GATED MULTIMODAL MODEL
# ============================================================

gated_model = GatedMultimodalPriceModel(
    image_dim=512,
    text_dim=512,
    structured_dim=STRUCTURED_DIM,
)

gated_model_path, gated_history = (
    train_model(
        gated_model,
        "gated_multimodal",
        gated=True,
    )
)


print()
print("=" * 80)
print("EXPERIMENTS COMPLETED")
print("=" * 80)

print(
    "Structured MLP:",
    structured_model_path,
)

print(
    "Gated Multimodal:",
    gated_model_path,
)

Using device: mps
Structured dim: 251
structured_mlp | Epoch 01/50 | Train Loss 0.3503 | Val Loss 0.0495 | Val MAE 15.9950 | Val RMSE 85.5857 | Val R² 0.4111 | LR 0.000500
  ✅ Best structured_mlp saved: 15.9950
structured_mlp | Epoch 02/50 | Train Loss 0.0920 | Val Loss 0.0615 | Val MAE 16.5560 | Val RMSE 87.5443 | Val R² 0.3838 | LR 0.000500
structured_mlp | Epoch 03/50 | Train Loss 0.0779 | Val Loss 0.0481 | Val MAE 15.2631 | Val RMSE 87.3214 | Val R² 0.3870 | LR 0.000500
  ✅ Best structured_mlp saved: 15.2631
structured_mlp | Epoch 04/50 | Train Loss 0.0751 | Val Loss 0.0374 | Val MAE 14.6501 | Val RMSE 84.2856 | Val R² 0.4288 | LR 0.000500
  ✅ Best structured_mlp saved: 14.6501
structured_mlp | Epoch 05/50 | Train Loss 0.0692 | Val Loss 0.0415 | Val MAE 15.2571 | Val RMSE 89.4027 | Val R² 0.3574 | LR 0.000500
structured_mlp | Epoch 06/50 | Train Loss 0.0652 | Val Loss 0.0758 | Val MAE 18.4393 | Val RMSE 90.5038 | Val R² 0.3415 | LR 0.000500
structured_mlp | Epoch 07/50 | Train Loss